# Module 4B: Interactive Presentation Demo Chatbot

This optional notebook provides a lightweight interactive Gradio interface over the same grounded evidence used by Module 4. It is designed for presentation use rather than for generating new scientific conclusions.

**Inputs:**  
- Module 2 pattern outputs  
- Module 3 evidence and analysis outputs  
- Module 4 transcript and quality-check outputs when available

**Processing steps:**  
1. Load available evidence files safely.  
2. Build an evidence index for retrieval.  
3. Detect question type from the user's input.  
4. Retrieve relevant evidence snippets.  
5. Generate template-based grounded answers.  
6. Optionally call an API-based chatbot mode when explicitly enabled.  
7. Save the interactive demo log for reproducibility.

**Role in the full pipeline:**  
Module 4B supports live presentation interaction while preserving the same evidence-grounded design as the validated Module 4 demo.

## Install and Import Dependencies

**Input:** Python runtime, optional `pandas`, and optional `gradio`.

**Processing:** Import standard libraries, use a lightweight pandas fallback if pandas is unavailable, and try to import Gradio. If Gradio is missing, the cell attempts a notebook-local pip install.

**Output:** Runtime imports, `pd` DataFrame/read_csv compatibility, and Gradio availability status.

In [ ]:
# ### module4B interactive demo chatbot cell 2
# This cell is documented for repository readability; comments describe the purpose without changing execution logic.

import csv
import json
import os
import subprocess
import sys
import textwrap
from datetime import datetime
from pathlib import Path
from typing import Any

try:
    import pandas as pd
    if not hasattr(pd, "DataFrame") or not hasattr(pd, "read_csv"):
        raise ImportError("pandas import is incomplete")
except Exception:
    class _MiniDataFrame:
        """Small DataFrame fallback for loading and displaying CSV-like rows."""
        def __init__(self, rows: list[dict[str, Any]] | None = None):
            self.rows = rows or []
        @property
        def columns(self) -> list[str]:
            cols = []
            for row in self.rows:
                for key in row:
                    if key not in cols:
                        cols.append(key)
            return cols
        def __len__(self) -> int:
            return len(self.rows)
        def __getitem__(self, key: str):
            return [row.get(key) for row in self.rows]
        def head(self, n: int = 5):
            return _MiniDataFrame(self.rows[:n])
        def to_dict(self, orient: str = "records"):
            return list(self.rows)
        def to_csv(self, path: Path, index: bool = False) -> None:
            with Path(path).open("w", encoding="utf-8", newline="") as file:
                writer = csv.DictWriter(file, fieldnames=self.columns)
                writer.writeheader()
                writer.writerows(self.rows)
        def __repr__(self) -> str:
            return json.dumps(self.rows[:10], indent=2, ensure_ascii=False)
    class _MiniPandas:
        DataFrame = _MiniDataFrame
        @staticmethod
        def read_csv(path: Path):
            with Path(path).open("r", encoding="utf-8", newline="") as file:
                return _MiniDataFrame(list(csv.DictReader(file)))
    pd = _MiniPandas()

try:
    import gradio as gr
    GRADIO_AVAILABLE = True
    GRADIO_IMPORT_MESSAGE = "gradio imported"
except Exception:
    print("Gradio is not installed. Attempting notebook-local install with pip...")
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "gradio", "-q"])
        import gradio as gr
        GRADIO_AVAILABLE = True
        GRADIO_IMPORT_MESSAGE = "gradio installed and imported"
    except Exception as exc:
        gr = None
        GRADIO_AVAILABLE = False
        GRADIO_IMPORT_MESSAGE = f"gradio unavailable: {type(exc).__name__}: {exc}"

print(GRADIO_IMPORT_MESSAGE)

## Define Project Paths

**Input:** Project root directory.

**Processing:** Define all module output paths with `pathlib.Path` and create the demo-only output directory.

**Output:** Reusable project paths and `module4B_demo_outputs/`.

In [ ]:
# ### module4B interactive demo chatbot cell 4
# This cell is documented for repository readability; comments describe the purpose without changing execution logic.

PROJECT_ROOT = Path.cwd()
MODULE2_DIR = PROJECT_ROOT / "module2_outputs"
MODULE3_DIR = PROJECT_ROOT / "module3_outputs"
MODULE4_DIR = PROJECT_ROOT / "module4_outputs"
DEMO_OUTPUT_DIR = PROJECT_ROOT / "module4B_demo_outputs"
DEMO_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Demo output directory: {DEMO_OUTPUT_DIR}")

## Safe File Loading Utilities

**Input:** Paths to existing JSON and CSV artifacts.

**Processing:** Load files defensively. Missing optional files return empty structures and print readable warnings.

**Output:** Helper functions for JSON, CSV, and demo-log saving.

In [ ]:
# ### module4B interactive demo chatbot cell 6
# This cell is documented for repository readability; comments describe the purpose without changing execution logic.

# ### Function: load_json
# Load a JSON artifact and raise a clear error if the file cannot be read.
def load_json(path: Path) -> dict:
    """Load a JSON file safely.

    Input: Path to a JSON file.
    Output: Parsed dictionary, or an empty dictionary when missing/unreadable.
    """
    if not path.exists():
        print(f"Warning: missing JSON file: {path}")
        return {}
    try:
        with path.open("r", encoding="utf-8") as file:
            obj = json.load(file)
        return obj if isinstance(obj, dict) else {"records": obj}
    except Exception as exc:
        print(f"Warning: could not load JSON {path}: {type(exc).__name__}: {exc}")
        return {}


# ### Function: load_csv_if_exists
# Define helper logic for load csv if exists used in this notebook step.
def load_csv_if_exists(path: Path):
    """Load a CSV file if it exists.

    Input: Path to a CSV file.
    Output: pandas-like DataFrame or None when missing/unreadable.
    """
    if not path.exists():
        print(f"Warning: missing CSV file: {path}")
        return None
    try:
        return pd.read_csv(path)
    except Exception as exc:
        print(f"Warning: could not load CSV {path}: {type(exc).__name__}: {exc}")
        return None


# ### Function: save_demo_log
# Save interactive chatbot interactions to disk.
def save_demo_log(log_records: list[dict], path: Path) -> None:
    """Save interactive demo log records to CSV.

    Input: Demo log records and target path.
    Output: Writes CSV when records exist; otherwise prints a concise message.
    """
    if not log_records:
        print("No demo interactions have been logged yet.")
        return
    fieldnames = ["timestamp", "user_question", "question_type", "answer_source", "quality_status", "warnings", "evidence_available", "answer_preview"]
    with path.open("w", encoding="utf-8", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        for record in log_records:
            row = record.copy()
            row["warnings"] = json.dumps(row.get("warnings", []), ensure_ascii=False)
            writer.writerow(row)
    print(f"Saved demo log: {path}")


# ### Function: table_records
# Convert a table-like object into record dictionaries.
def table_records(table) -> list[dict]:
    """Convert a pandas-like table into list-of-dict records.

    Input: pandas DataFrame, fallback DataFrame, list, or None.
    Output: List of row dictionaries.
    """
    if table is None:
        return []
    if hasattr(table, "to_dict"):
        return table.to_dict(orient="records")
    return list(table) if isinstance(table, list) else []

## Load Existing Evidence

**Input:** Existing Module 2, Module 3, and Module 4 output files.

**Processing:** Load JSON and CSV evidence artifacts if available and build a compact status table.

**Output:** Loaded evidence objects and file-loading status table.

In [ ]:
# ### module4B interactive demo chatbot cell 8
# This cell is documented for repository readability; comments describe the purpose without changing execution logic.

EVIDENCE_FILES = {
    "module2_llm_ready_patterns.json": MODULE2_DIR / "module2_llm_ready_patterns.json",
    "module2_pattern_table_with_sentences.csv": MODULE2_DIR / "module2_pattern_table_with_sentences.csv",
    "module2_classifier_summary.csv": MODULE2_DIR / "module2_classifier_summary.csv",
    "module3B_evidence_pack.json": MODULE3_DIR / "module3B_evidence_pack.json",
    "module3D_llm_analysis.json": MODULE3_DIR / "module3D_llm_analysis.json",
    "module3E_grounding_summary.json": MODULE3_DIR / "module3E_grounding_summary.json",
    "module4_api_chatbot_responses.json": MODULE4_DIR / "module4_api_chatbot_responses.json",
    "module4_answer_quality_summary.json": MODULE4_DIR / "module4_answer_quality_summary.json",
}

llm_ready_patterns = load_json(EVIDENCE_FILES["module2_llm_ready_patterns.json"])
pattern_table = load_csv_if_exists(EVIDENCE_FILES["module2_pattern_table_with_sentences.csv"])
classifier_summary = load_csv_if_exists(EVIDENCE_FILES["module2_classifier_summary.csv"])
evidence_pack = load_json(EVIDENCE_FILES["module3B_evidence_pack.json"])
llm_analysis = load_json(EVIDENCE_FILES["module3D_llm_analysis.json"])
grounding_summary = load_json(EVIDENCE_FILES["module3E_grounding_summary.json"])
api_chatbot_responses = load_json(EVIDENCE_FILES["module4_api_chatbot_responses.json"])
module4_quality_summary = load_json(EVIDENCE_FILES["module4_answer_quality_summary.json"])

_loaded_objects = {
    "module2_llm_ready_patterns.json": llm_ready_patterns,
    "module2_pattern_table_with_sentences.csv": pattern_table,
    "module2_classifier_summary.csv": classifier_summary,
    "module3B_evidence_pack.json": evidence_pack,
    "module3D_llm_analysis.json": llm_analysis,
    "module3E_grounding_summary.json": grounding_summary,
    "module4_api_chatbot_responses.json": api_chatbot_responses,
    "module4_answer_quality_summary.json": module4_quality_summary,
}

load_status_rows = []
for name, obj in _loaded_objects.items():
    if obj is None:
        loaded, obj_type, count = False, "None", 0
    elif hasattr(obj, "to_dict"):
        loaded, obj_type, count = True, "table", len(obj)
    elif isinstance(obj, dict):
        loaded, obj_type, count = bool(obj), "dict", len(obj.keys())
    else:
        loaded, obj_type, count = True, type(obj).__name__, 0
    load_status_rows.append({"file_name": name, "loaded": loaded, "object_type": obj_type, "rows_or_keys": count})

load_status_table = pd.DataFrame(load_status_rows)
display(load_status_table)

## Build Evidence Index

**Input:** Loaded pattern table, classifier summary, evidence pack, grounding summary, and LLM analysis.

**Processing:** Build compact evidence categories for common presentation questions. Module 3B evidence is preferred, with Module 2 and Module 3D/3E fallbacks.

**Output:** `evidence_index`, a compact dictionary used by the chatbot.

In [ ]:
# ### module4B interactive demo chatbot cell 10
# This cell is documented for repository readability; comments describe the purpose without changing execution logic.

# ### Function: _safe_items
# Return list-like evidence safely even when fields are missing.
def _safe_items(value: Any) -> list[dict]:
    """Return list-like evidence records safely.

    Input: Any value from an evidence artifact.
    Output: List of dictionaries when available.
    """
    return value if isinstance(value, list) else []


# ### Function: _select_fields
# Keep only selected fields from one record for compact display.
def _select_fields(row: dict, fields: list[str]) -> dict:
    """Select available fields from a record.

    Input: Source row and desired field names.
    Output: Compact dictionary containing available fields only.
    """
    return {field: row.get(field) for field in fields if field in row and row.get(field) not in (None, "")}


# ### Function: build_evidence_index
# Build an indexed collection of evidence snippets and summary records.
def build_evidence_index(
    pattern_table: Any,
    classifier_summary: Any,
    evidence_pack: dict,
    grounding_summary: dict,
    llm_analysis: dict,
) -> dict:
    """Build compact evidence categories for the interactive chatbot.

    Input: Loaded Module 2 and Module 3 evidence artifacts.
    Output: Dictionary with presentation-friendly evidence categories.
    """
    pattern_fields = ["classifier", "normalization", "error_50", "error_70", "error_80", "error_90", "error_100", "delta_100_50", "relative_increase_pct", "trend_label", "robustness_flag", "spike_type", "curve_shape", "pattern_strength", "degradation_type", "pattern_sentence"]
    pattern_records = [_select_fields(row, pattern_fields) for row in table_records(pattern_table)]
    classifier_records = table_records(classifier_summary)
    sensitive = _safe_items(evidence_pack.get("most_sensitive_combinations")) or sorted(pattern_records, key=lambda row: float(row.get("delta_100_50", 0) or 0), reverse=True)[:5]
    stable = _safe_items(evidence_pack.get("most_stable_combinations")) or sorted(pattern_records, key=lambda row: float(row.get("delta_100_50", 999) or 999))[:5]
    classifier_evidence = _safe_items(evidence_pack.get("classifier_level_evidence")) or classifier_records
    normalization_evidence = _safe_items(evidence_pack.get("normalization_level_evidence"))
    scenario_overview = evidence_pack.get("scenario_overview", {}) if isinstance(evidence_pack, dict) else {}
    project_scope = evidence_pack.get("project_scope", {}) if isinstance(evidence_pack, dict) else {}
    return {
        "classifier_sensitivity": classifier_evidence,
        "normalization_stability": normalization_evidence,
        "split_trend": {"scenario_overview": scenario_overview, "project_scope": project_scope, "representative_patterns": pattern_records[:5]},
        "most_sensitive_combinations": sensitive[:5],
        "most_stable_combinations": stable[:5],
        "llm_grounding_quality": {"grounding_summary": grounding_summary, "quality_summary": module4_quality_summary},
        "limitations": {"analysis_scope_note": project_scope.get("analysis_scope_note"), "llm_limitations": llm_analysis.get("limitations"), "constraints": project_scope.get("explicit_constraints", [])},
        "overall_summary": {"scenario_overview": scenario_overview, "llm_analysis": llm_analysis, "most_sensitive": sensitive[:3], "most_stable": stable[:3]},
    }


evidence_index = build_evidence_index(pattern_table, classifier_summary, evidence_pack, grounding_summary, llm_analysis)
print("Evidence index categories:", list(evidence_index.keys()))

## Question Type Detection

**Input:** User question text.

**Processing:** Classify the question into a supported evidence category using deterministic keyword logic.

**Output:** Question type string used for evidence retrieval.

In [ ]:
# ### module4B interactive demo chatbot cell 12
# This cell is documented for repository readability; comments describe the purpose without changing execution logic.

# ### Function: detect_question_type
# Classify the user question into a broad answer category.
def detect_question_type(question: str) -> str:
    """Classify a natural-language question into a supported demo category.

    Input: User question.
    Output: One question type label.
    """
    q = question.lower()
    if any(term in q for term in ["grounding", "consistent", "quality", "llm output", "llm-generated", "llm generated"]):
        return "llm_grounding_quality"
    if any(term in q for term in ["limitation", "weakness", "caveat"]):
        return "limitations"
    if any(term in q for term in ["most sensitive", "worst", "largest delta"]):
        return "most_sensitive_combinations"
    if any(term in q for term in ["most robust", "most stable", "best", "robust combinations"]):
        return "most_stable_combinations"
    if any(term in q for term in ["split", "50", "100", "increase", "cross-batch", "cross batch"]):
        return "split_trend"
    if any(term in q for term in ["normalization", "stable", "non", "qn", "mn", "vsn"]):
        return "normalization_stability"
    if any(term in q for term in ["classifier", "model", "sensitive", "batch-sensitive", "batch sensitive"]):
        return "classifier_sensitivity"
    if any(term in q for term in ["summary", "main finding", "main findings", "overall"]):
        return "overall_summary"
    return "unknown"

## Evidence Retrieval

**Input:** User question and evidence index.

**Processing:** Detect question type and retrieve compact, presentation-friendly evidence items.

**Output:** Retrieved evidence dictionary with availability status and fallback message.

In [ ]:
# ### module4B interactive demo chatbot cell 14
# This cell is documented for repository readability; comments describe the purpose without changing execution logic.

# ### Function: retrieve_relevant_evidence
# Retrieve the most relevant evidence for one question.
def retrieve_relevant_evidence(question: str, evidence_index: dict, max_items: int = 5) -> dict:
    """Retrieve compact evidence for a user question.

    Input: User question, evidence index, and maximum number of list items.
    Output: Retrieval dictionary with question type and evidence items.
    """
    question_type = detect_question_type(question)
    evidence = evidence_index.get(question_type)
    if not evidence:
        return {"question_type": question_type, "evidence_items": [], "evidence_available": False, "fallback_message": "No matching loaded evidence was found for this question type."}
    if isinstance(evidence, list):
        items = evidence[:max_items]
    elif isinstance(evidence, dict):
        items = evidence
    else:
        items = [evidence]
    return {"question_type": question_type, "evidence_items": items, "evidence_available": bool(items), "fallback_message": ""}


# ### Function: format_evidence_for_display
# Format retrieved evidence for display in the interface.
def format_evidence_for_display(evidence_items: Any, max_chars: int = 1200) -> str:
    """Format retrieved evidence compactly for display.

    Input: Evidence items and maximum character count.
    Output: Readable compact evidence string.
    """
    text = json.dumps(evidence_items, indent=2, ensure_ascii=False)
    return text if len(text) <= max_chars else text[:max_chars] + "\n..."

## Template-Based Grounded Answer Generator

**Input:** User question and retrieved evidence.

**Processing:** Generate a concise deterministic answer grounded only in loaded project evidence.

**Output:** Four-part answer with direct answer, key evidence, interpretation, and caveat.

In [ ]:
# ### module4B interactive demo chatbot cell 16
# This cell is documented for repository readability; comments describe the purpose without changing execution logic.

# ### Function: _pair_label
# Create a readable classifier-normalization label.
def _pair_label(item: dict) -> str:
    """Format classifier-normalization pair label.

    Input: Combination evidence dictionary.
    Output: Label such as knn|vsn.
    """
    return f"{item.get('classifier', 'NA')}|{item.get('normalization', 'NA')}"


# ### Function: generate_template_answer
# Generate a deterministic grounded answer from retrieved evidence.
def generate_template_answer(question: str, retrieved: dict) -> str:
    """Generate a deterministic grounded answer from retrieved evidence.

    Input: User question and retrieved evidence dictionary.
    Output: Concise grounded answer text.
    """
    qtype = retrieved.get("question_type", "unknown")
    evidence = retrieved.get("evidence_items")
    if not retrieved.get("evidence_available"):
        return (
            "Direct answer: The loaded evidence is insufficient for that specific question.\n\n"
            "Key evidence: No matching evidence category was available.\n\n"
            "Interpretation: Try asking about classifier sensitivity, normalization stability, split trends, sensitive/robust combinations, grounding quality, limitations, or an overall summary.\n\n"
            "Caveat: This chatbot only answers from loaded project evidence."
        )

    if qtype == "most_sensitive_combinations":
        top = evidence[0] if isinstance(evidence, list) and evidence else {}
        direct = f"The most sensitive combination appears to be {_pair_label(top)}, based on the largest delta_100_50 available in the extracted evidence."
        key = f"Top evidence: {_pair_label(top)} with delta_100_50={top.get('delta_100_50')} and robustness_flag={top.get('robustness_flag')}."
    elif qtype == "most_stable_combinations":
        top = evidence[0] if isinstance(evidence, list) and evidence else {}
        direct = f"The most robust or stable combination appears to be {_pair_label(top)}, based on the smallest available delta_100_50."
        key = f"Top evidence: {_pair_label(top)} with delta_100_50={top.get('delta_100_50')} and degradation_type={top.get('degradation_type')}."
    elif qtype == "classifier_sensitivity":
        records = evidence if isinstance(evidence, list) else []
        top = sorted(records, key=lambda row: float(row.get("mean_delta_100_50", 0) or 0), reverse=True)[0] if records else {}
        direct = f"The classifier that appears most batch-sensitive is {top.get('classifier', 'not available')}."
        key = top.get("classifier_summary_sentence", json.dumps(top, ensure_ascii=False))
    elif qtype == "normalization_stability":
        records = evidence if isinstance(evidence, list) else []
        top = sorted(records, key=lambda row: float(row.get("mean_delta_100_50", 999) or 999))[0] if records else {}
        direct = f"The normalization method that appears most stable is {top.get('normalization', 'not available')}."
        key = f"Its mean_delta_100_50 is {top.get('mean_delta_100_50')} with dominant_robustness_flag={top.get('dominant_robustness_flag')}."
    elif qtype == "split_trend":
        overview = evidence.get("scenario_overview", {}) if isinstance(evidence, dict) else {}
        direct = "As split changes from 50 to 100, classification error generally appears to increase in the extracted error-curve evidence."
        key = overview.get("dominant_observation", json.dumps(overview, ensure_ascii=False))
    elif qtype == "llm_grounding_quality":
        gs = evidence.get("grounding_summary", {}) if isinstance(evidence, dict) else {}
        direct = f"The LLM interpretation appears broadly grounded with status {gs.get('overall_grounding_status', 'not available')}."
        key = f"Classifier coverage={gs.get('classifier_coverage_rate')}; normalization coverage={gs.get('normalization_coverage_rate')}."
    elif qtype == "limitations":
        direct = "The main limitations are that the findings are limited to the loaded scenario, classifiers, normalizations, split values, and classification error evidence."
        key = evidence.get("analysis_scope_note") or evidence.get("llm_limitations") or json.dumps(evidence, ensure_ascii=False)
    else:
        direct = "The main finding is that stronger batch-separated evaluation is associated with higher classification error for many classifier-normalization curves."
        key = format_evidence_for_display(evidence, max_chars=500)
    return (
        f"Direct answer: {direct}\n\n"
        f"Key evidence: {key}\n\n"
        "Interpretation: This suggests a project-specific robustness pattern based on the extracted error-curve evidence, not on general knowledge.\n\n"
        "Caveat: This answer is based only on loaded project evidence and does not claim causality or infer biological mechanisms."
    )

## Optional API-Based Answer Generator

**Input:** User question, retrieved evidence, and optional `OPENAI_API_KEY` if API mode is manually enabled.

**Processing:** Default to template mode. If enabled and available, call an API using retrieved evidence only; otherwise fall back to the template answer.

**Output:** Grounded answer text and answer source label.

In [ ]:
# ### module4B interactive demo chatbot cell 18
# This cell is documented for repository readability; comments describe the purpose without changing execution logic.

USE_API_CHATBOT = False
OPENAI_MODEL = "gpt-4o-mini"


# ### Function: generate_api_answer
# Generate an API-based answer constrained by retrieved evidence.
def generate_api_answer(question: str, retrieved: dict) -> str:
    """Generate an optional API answer from retrieved evidence only.

    Input: User question and retrieved evidence.
    Output: API answer text, or raises an exception for fallback handling.
    """
    if not os.environ.get("OPENAI_API_KEY"):
        raise RuntimeError("OPENAI_API_KEY is not set")
    from openai import OpenAI
    client = OpenAI()
    prompt = (
        "Answer only from the provided project evidence. Do not invent numbers. "
        "Do not claim causality. Include a caveat. Keep the answer concise.\n\n"
        f"Question: {question}\n\nRetrieved evidence:\n{json.dumps(retrieved, indent=2, ensure_ascii=False)}"
    )
    response = client.responses.create(model=OPENAI_MODEL, input=prompt, temperature=0, max_output_tokens=500)
    return response.output_text


# ### Function: generate_grounded_answer
# Choose API generation or template fallback and return the answer source.
def generate_grounded_answer(question: str, retrieved: dict) -> tuple[str, str]:
    """Generate either an API answer or template fallback.

    Input: User question and retrieved evidence.
    Output: Tuple of answer text and answer source label.
    """
    template_answer = generate_template_answer(question, retrieved)
    if not USE_API_CHATBOT:
        return template_answer, "template"
    try:
        return generate_api_answer(question, retrieved), "api"
    except Exception:
        return template_answer, "template_fallback"

## Lightweight Quality Check

**Input:** Generated answer and retrieved evidence metadata.

**Processing:** Check answer presence, evidence availability, cautious wording, and forbidden unsupported terms.

**Output:** Lightweight quality status dictionary.

In [ ]:
# ### module4B interactive demo chatbot cell 20
# This cell is documented for repository readability; comments describe the purpose without changing execution logic.

# ### Function: run_lightweight_quality_check
# Run lightweight checks on one interactive answer.
def run_lightweight_quality_check(answer: str, retrieved: dict) -> dict:
    """Run a lightweight deterministic answer quality check.

    Input: Answer text and retrieved evidence dictionary.
    Output: Quality status dictionary.
    """
    warnings = []
    answer_lower = answer.lower()
    if not answer.strip():
        return {"status": "fail", "warnings": ["empty answer"], "evidence_available": retrieved.get("evidence_available", False), "question_type": retrieved.get("question_type", "unknown")}
    if not retrieved.get("evidence_available", False):
        warnings.append("no evidence available for detected question type")
    if not any(term in answer_lower for term in ["suggests", "appears", "consistent", "based on"]):
        warnings.append("answer lacks cautious interpretation wording")
    forbidden = [term for term in ["proves", "causes", "guarantees", "definitively", "certainly"] if term in answer_lower]
    if forbidden:
        warnings.append(f"forbidden unsupported terms: {forbidden}")
    status = "fail" if forbidden else ("warning" if warnings else "pass")
    return {"status": status, "warnings": warnings, "evidence_available": retrieved.get("evidence_available", False), "question_type": retrieved.get("question_type", "unknown")}

## Main Chatbot Function

**Input:** A user message from Gradio or a notebook test call.

**Processing:** Retrieve evidence, generate a grounded answer, run a quality check, and append an in-memory demo log record.

**Output:** Formatted response containing answer, supporting evidence, and quality status.

In [ ]:
# ### module4B interactive demo chatbot cell 22
# This cell is documented for repository readability; comments describe the purpose without changing execution logic.

DEMO_LOG: list[dict[str, Any]] = []


# ### Function: chatbot_response
# Main interactive chatbot function used by Gradio.
def chatbot_response(message: str, history: list | None = None) -> str:
    """Return a grounded chatbot response for one user message.

    Input: User message and optional chat history.
    Output: Formatted answer for display in the chat interface.
    """
    retrieved = retrieve_relevant_evidence(message, evidence_index)
    answer, answer_source = generate_grounded_answer(message, retrieved)
    quality = run_lightweight_quality_check(answer, retrieved)
    evidence_text = format_evidence_for_display(retrieved.get("evidence_items"), max_chars=900)
    DEMO_LOG.append({
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "user_question": message,
        "question_type": retrieved.get("question_type"),
        "answer_source": answer_source,
        "quality_status": quality.get("status"),
        "warnings": quality.get("warnings", []),
        "evidence_available": quality.get("evidence_available"),
        "answer_preview": answer[:240],
    })
    notes = "; ".join(quality.get("warnings", [])) if quality.get("warnings") else "None"
    return f"Answer:\n{answer}\n\nSupporting evidence:\n```json\n{evidence_text}\n```\n\nQuality check:\nStatus: {quality.get('status', '').upper()}\nNotes: {notes}"

sample_chatbot_response = chatbot_response("Which classifier is most batch-sensitive?")
print(sample_chatbot_response[:700])

## Demo Question Buttons / Examples

**Input:** Common presentation questions.

**Processing:** Define example prompts for the Gradio interface.

**Output:** `EXAMPLE_QUESTIONS` list.

In [ ]:
# ### module4B interactive demo chatbot cell 24
# This cell is documented for repository readability; comments describe the purpose without changing execution logic.

EXAMPLE_QUESTIONS = [
    "Which classifier is most batch-sensitive?",
    "Which normalization method appears most stable?",
    "What happens when split changes from 50 to 100?",
    "What are the most robust classifier-normalization combinations?",
    "What are the most sensitive classifier-normalization combinations?",
    "Is the LLM-generated interpretation consistent with the observed evidence?",
    "What are the main limitations of this experiment?",
    "Summarize the main findings of the project.",
]
print(f"Example questions configured: {len(EXAMPLE_QUESTIONS)}")

## Launch Gradio Interface

**Input:** Chatbot function and example questions.

**Processing:** Create and launch a Gradio chat interface. Use `share=True` for Colab/presentation sharing; set `share=False` for local-only use.

**Output:** Live notebook chatbot interface when this cell is executed.

In [ ]:
# ### module4B interactive demo chatbot cell 26
# This cell is documented for repository readability; comments describe the purpose without changing execution logic.

if GRADIO_AVAILABLE:
    demo = gr.ChatInterface(
        fn=chatbot_response,
        title="Interactive Demo: LLM-Assisted ML Result Analysis",
        description="Ask questions about classifier robustness, normalization stability, batch-sensitive error trends, and LLM output grounding. The chatbot answers using structured evidence generated by earlier project modules.",
        examples=EXAMPLE_QUESTIONS,
    )
    # For local-only demos, change share=True to share=False.
    demo.launch(share=True, debug=False)
else:
    demo = None
    print("Gradio is unavailable in this environment. Run the install/import cell again or install gradio before launching the interface.")

## Save Demo Log

**Input:** In-memory `DEMO_LOG` records from interactive use.

**Processing:** Save interactions if any have been logged.

**Output:** Optional `module4B_demo_outputs/module4B_interactive_demo_log.csv`.

In [ ]:
# ### module4B interactive demo chatbot cell 28
# This cell is documented for repository readability; comments describe the purpose without changing execution logic.

save_demo_log(DEMO_LOG, DEMO_OUTPUT_DIR / "module4B_interactive_demo_log.csv")

## Final Notebook Checklist

**Input:** Loaded evidence state, evidence index, chatbot function, Gradio interface state, and output directory.

**Processing:** Print concise readiness checks and save a JSON summary for the interactive demo notebook.

**Output:** `module4B_demo_outputs/module4B_demo_summary.json`.

In [ ]:
# ### module4B interactive demo chatbot cell 30
# This cell is documented for repository readability; comments describe the purpose without changing execution logic.

loaded_file_summary = {row["file_name"]: row["loaded"] for row in load_status_rows}
checklist = {
    "evidence_files_loaded": any(loaded_file_summary.values()),
    "evidence_index_built": bool(evidence_index),
    "chatbot_function_available": callable(chatbot_response),
    "gradio_interface_created": GRADIO_AVAILABLE and "demo" in globals() and demo is not None,
    "demo_output_directory_exists": DEMO_OUTPUT_DIR.exists(),
    "no_previous_module_outputs_overwritten": True,
}
summary = {
    "notebook_name": "module4B_interactive_demo_chatbot.ipynb",
    "purpose": "Interactive presentation demo chatbot using saved structured evidence.",
    "loaded_files": loaded_file_summary,
    "demo_interface": "gradio_chatinterface" if checklist["gradio_interface_created"] else "gradio_unavailable_or_not_launched",
    "uses_api_by_default": False,
    "output_directory": str(DEMO_OUTPUT_DIR),
    "status": "ready" if checklist["evidence_index_built"] and checklist["chatbot_function_available"] else "incomplete",
}
with (DEMO_OUTPUT_DIR / "module4B_demo_summary.json").open("w", encoding="utf-8") as file:
    json.dump(summary, file, indent=2, ensure_ascii=False)

for key, value in checklist.items():
    print(f"{key}: {value}")
print(f"Saved summary: {DEMO_OUTPUT_DIR / 'module4B_demo_summary.json'}")

## Final Presentation Note

To visibly demonstrate the real-time chatbot interface, run the Gradio launch cell in a local Jupyter notebook or Colab session. In Colab or presentation mode, `share=True` can create a public demo link; for local-only use, change it to `share=False`.

The official project outputs remain the saved Module 1-4 artifacts. This notebook is a presentation layer that reads those artifacts and writes only demo-layer files under `module4B_demo_outputs/`.